# KV-cache selection research log (cleaned source)

Historical project cells from the uploaded notebook. Outputs and execution metadata have been cleared.

**For a fresh run, use `reproduce.ipynb` instead.** This research log intentionally retains historical run IDs and the original Modal paths; it is not a portable Run All notebook. Do not use edited files to resume old hash-locked experiments.

The unrelated assignment cells at the end of the uploaded notebook were excluded from this public release. Model prompts, answer labels, and experimental rules in the retained cells were not changed.


In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("gstar")


## Original project cell 1


In [ ]:
import sys
import torch
from importlib.metadata import version, PackageNotFoundError

logger.debug('%s %s', "Python:", sys.version.split()[0])
logger.debug('%s %s', "PyTorch:", torch.__version__)
logger.debug('%s %s', "CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. Check the notebook kernel GPU configuration."
    )

logger.debug('%s %s', "GPU:", torch.cuda.get_device_name(0))
logger.debug('%s %s', "GPU count:", torch.cuda.device_count())

gpu = torch.cuda.get_device_properties(0)
logger.debug('%s', f"GPU memory: {gpu.total_memory / 1024**3:.1f} GiB")

try:
    logger.debug('%s %s', "Transformers:", version("transformers"))
except PackageNotFoundError:
    logger.debug('%s', "Transformers is not installed.")

## Original project cell 2


In [ ]:
from pathlib import Path

mnt = Path("/mnt")

if not mnt.is_dir():
    logger.debug('%s', "The /mnt directory is missing. Check the volume mount.")
else:
    folders = sorted(p for p in mnt.iterdir() if p.is_dir())

    if not folders:
        logger.debug('%s', "No directories found under /mnt. Check the volume mount.")
    else:
        logger.debug('%s', "Directories under /mnt:")
        for folder in folders:
            logger.debug('%s', folder)

## Original project cell 3


In [ ]:
from pathlib import Path
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_DIR = Path("/mnt/gstar-project")
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

if not PROJECT_DIR.is_dir():
    raise RuntimeError("Check that the project directory or volume is available.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable.")

CACHE_DIR = PROJECT_DIR / "hf_cache"
CACHE_DIR.mkdir(exist_ok=True)

logger.debug('%s %s', "Model cache directory:", CACHE_DIR)
logger.debug('%s', "Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=str(CACHE_DIR),
)

logger.debug('%s', "Downloading or loading model...")
started = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=str(CACHE_DIR),
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    use_safetensors=True,
)
model = model.to("cuda")
model.eval()

torch.cuda.synchronize()

logger.debug('%s', f"\nModel loaded. Elapsed time: {time.perf_counter() - started:.1f}seconds")
logger.debug('%s %s', "Model device:", model.device)
logger.debug('%s %s', "Model dtype:", model.dtype)
logger.debug('%s', f"Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

## Original project cell 4


In [ ]:
from transformers import GenerationConfig

context = (
    "The silver cabinet has access code 7316. "
    "The blue cabinet has access code 2849."
)
question = "What is the access code for the silver cabinet?"

messages = [
    {
        "role": "system",
        "content": (
            "Answer the question using only the provided context. "
            "Return only the answer, without explanation."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion:\n{question}",
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Use greedy decoding without sampling.
gen_config = GenerationConfig(
    do_sample=False,
    max_new_tokens=32,
    use_cache=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=model.generation_config.eos_token_id,
)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        generation_config=gen_config,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

logger.debug('%s %s', "Question:", question)
logger.debug('%s %s', "Model answer:", answer)
logger.debug('%s', "Expected answer: 7316")

## Original project cell 5


In [ ]:
import copy
import torch
from transformers import GenerationConfig

test_context = (
    "The silver cabinet has access code 7316. "
    "The blue cabinet has access code 2849."
)

system_prompt = (
    "Answer the question using only the provided context. "
    "Return only the answer, without explanation."
)

def render_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"Context:\n{test_context}\n\nQuestion:\n{question}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# Use a temporary marker to locate the start of the actual question.
# The marker itself is not passed to the model.
marker = "__GSTAR_QUESTION_SLOT__"
template_text = render_prompt(marker)
assert template_text.count(marker) == 1

prefix_text = template_text.split(marker, 1)[0]

prefix_inputs = tokenizer(
    prefix_text,
    add_special_tokens=False,
    return_tensors="pt",
).to(model.device)

model.eval()

# Prefill only the document and fixed prompt, without the actual question.
with torch.no_grad():
    prefix_output = model(
        **prefix_inputs,
        use_cache=True,
        logits_to_keep=1,
    )

document_cache = prefix_output.past_key_values
prefix_length = prefix_inputs["input_ids"].shape[1]
del prefix_output

assert document_cache.get_seq_length() == prefix_length

logger.debug('%s', "Document KV cache created.")
logger.debug('%s %s', "Shared prefix tokens:", prefix_length)
logger.debug('%s %s', "Cache type:", type(document_cache).__name__)

## Original project cell 6


In [ ]:
check_config = GenerationConfig(
    do_sample=False,
    max_new_tokens=32,
    use_cache=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=model.generation_config.eos_token_id,
)

test_cases = [
    ("What is the access code for the silver cabinet?", "7316"),
    ("What is the access code for the blue cabinet?", "2849"),
    ("What is the access code for the silver cabinet?", "7316"),
]

reuse_test_results = []

for i, (question, expected) in enumerate(test_cases, start=1):
    full_inputs = tokenizer(
        render_prompt(question),
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)

    # Check equality of prefix token IDs, not just the rendered strings.
    assert torch.equal(
        full_inputs["input_ids"][:, :prefix_length],
        prefix_inputs["input_ids"],
    ), "Prefix tokens changed. Check the input split."

    input_length = full_inputs["input_ids"].shape[1]

    with torch.no_grad():
        # A. Reference path: compute the full document and question from scratch.
        fresh_output = model.generate(
            **full_inputs,
            generation_config=check_config,
        )

        # B. Reuse path: start from an independent copy of the pristine document cache.
        reused_output = model.generate(
            **full_inputs,
            past_key_values=copy.deepcopy(document_cache),
            generation_config=check_config,
        )

    fresh_tokens = fresh_output[0, input_length:]
    reused_tokens = reused_output[0, input_length:]

    fresh_answer = tokenizer.decode(
        fresh_tokens, skip_special_tokens=True
    ).strip()
    reused_answer = tokenizer.decode(
        reused_tokens, skip_special_tokens=True
    ).strip()

    tokens_match = torch.equal(fresh_tokens, reused_tokens)

    # Ensure question and answer tokens were not appended to the original cache.
    assert document_cache.get_seq_length() == prefix_length

    reuse_test_results.append({
        "question": question,
        "expected": expected,
        "fresh_answer": fresh_answer,
        "reused_answer": reused_answer,
        "tokens_match": tokens_match,
    })

    logger.debug('%s', f"\n[Question {i}] {question}")
    logger.debug('%s %s', "Fresh computation:", fresh_answer)
    logger.debug('%s %s', "Reused document KV:", reused_answer)
    logger.debug('%s %s', "Expected answer:", expected)
    logger.debug('%s %s', "Generated tokens identical:", tokens_match)
    logger.debug('%s %s', "Original cache length preserved:", True)

## Original project cell 7


In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path("/mnt/gstar-project")
ARCHIVE = PROJECT / "gstar_fullkv_pilot.zip"

if not ARCHIVE.is_file():
    raise FileNotFoundError(
        f"{ARCHIVE} is missing. Upload the ZIP archive first."
    )

with zipfile.ZipFile(ARCHIVE) as z:
    # Prevent extraction outside the project directory.
    for item in z.infolist():
        destination = (PROJECT / item.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError("The ZIP archive contains an unsafe path.")

    z.extractall(PROJECT)

logger.info('%s %s', "Prepared:", PROJECT / "gstar_fullkv_pilot")

## Original project cell 8


In [ ]:
import runpy

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the model-loading cell first."
    )

pilot = runpy.run_path(
    str(PROJECT / "gstar_fullkv_pilot" / "gstar_run_fullkv.py")
)

pilot_summary = pilot["run_fullkv_pilot"](
    model=model,
    tokenizer=tokenizer,
    project_dir=PROJECT,
)

## Original project cell 9


In [ ]:
import json
from pathlib import Path

project = Path("/mnt/gstar-project")

# Locate the data used by the most recent completed run.
env_files = sorted(
    (project / "results").glob("fullkv_pilot_*/environment.json")
)
if not env_files:
    raise FileNotFoundError("No saved environment.json was found.")

env_path = env_files[-1]
env = json.loads(env_path.read_text(encoding="utf-8"))
data_file = Path(env["data_directory"]) / "selection_inputs.jsonl"

if not data_file.is_file():
    raise FileNotFoundError(f"Data file not found: {data_file}")

rows = [
    json.loads(line)
    for line in data_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
matches = [r for r in rows if r["document_id"] == "pilot_003"]

if not matches:
    raise ValueError("pilot_003 is absent from the data.")

# Verify that both conditions use the same document.
document = matches[0]["document"]
assert all(r["document"] == document for r in matches)

logger.debug('%s %s', "Inspected run:", env_path.parent.name)
logger.debug('%s', "\n========== pilot_003 source document ==========\n")
logger.debug('%s', document)

## Original project cell 10


In [ ]:
import json
import runpy
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers

project = Path("/mnt/gstar-project")

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError("The current kernel must contain the previously loaded model/tokenizer.")

# 1. Reuse the exact data and prompt from the preceding Full-KV experiment.
env_files = sorted(
    (project / "results").glob("fullkv_pilot_*/environment.json")
)
if not env_files:
    raise FileNotFoundError("The Full-KV environment.json file is missing.")

env_path = env_files[-1]
env = json.loads(env_path.read_text(encoding="utf-8"))
selection_path = Path(env["data_directory"]) / "selection_inputs.jsonl"

selection_rows = [
    json.loads(line)
    for line in selection_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
case = next(r for r in selection_rows if r["case_id"] == "pilot_001_A")
document = case["document"]
question = case["first_question"]

runner = runpy.run_path(
    str(project / "gstar_fullkv_pilot" / "gstar_run_fullkv.py")
)
render = runner["render_prompt"]

# 2. Locate the character spans of the document and first question.
marker = "__GSTAR_QUESTION_SLOT__"
marked_prompt = render(tokenizer, document, marker)
assert marked_prompt.count(marker) == 1
prefix_text, suffix_text = marked_prompt.split(marker)

prompt = render(tokenizer, document, question)
assert prompt == prefix_text + question + suffix_text
assert prompt.count(document) == 1

doc_start = prompt.index(document)
doc_end = doc_start + len(document)
q_start = len(prefix_text)
q_end = q_start + len(question)

# Map character offsets to model token positions.
if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for token offset mapping.")

encoded = tokenizer(
    prompt,
    add_special_tokens=False,
    return_offsets_mapping=True,
    return_tensors="pt",
)
offsets = encoded.pop("offset_mapping")[0].tolist()

def overlapping_positions(start, end):
    return [
        i for i, (s, e) in enumerate(offsets)
        if e > s and s < end and e > start
    ]

doc_positions = overlapping_positions(doc_start, doc_end)
question_positions = overlapping_positions(q_start, q_end)
assert doc_positions and question_positions
assert max(doc_positions) < min(question_positions)

# This short diagnostic materializes full attention; reject long inputs here.
input_length = encoded["input_ids"].shape[1]
if input_length > 1024:
    raise ValueError("This diagnostic supports at most 1024 tokens and does not truncate inputs.")

document_token_ids = encoded["input_ids"][0, doc_positions].tolist()
encoded = encoded.to(model.device)
d_idx = torch.tensor(doc_positions, device=model.device)
q_idx = torch.tensor(question_positions, device=model.device)

# 3. Extract first-question-to-document attention and average by KV head.
def extract_raw_a():
    old_backend = model.config._attn_implementation
    if not old_backend or not hasattr(model, "set_attn_implementation"):
        raise RuntimeError("The current attention backend must be checked.")

    hq = model.config.num_attention_heads
    hkv = model.config.num_key_value_heads
    assert hq % hkv == 0

    model.eval()
    try:
        model.set_attn_implementation("eager")

        with torch.no_grad():
            output = model(
                **encoded,
                use_cache=False,
                output_attentions=True,
                return_dict=True,
                logits_to_keep=1,
            )

        if not output.attentions or any(x is None for x in output.attentions):
            raise RuntimeError("Attention weights were not returned.")

        per_layer = []
        for weights in output.attentions:
            # [query heads, first-question tokens, document tokens]
            block = (
                weights[0]
                .index_select(1, q_idx)
                .index_select(2, d_idx)
                .float()
            )
            assert block.shape[0] == hq

            # [KV heads, grouped query heads, question tokens, document tokens]
            grouped = block.reshape(
                hkv, hq // hkv, len(question_positions), len(doc_positions)
            )
            per_layer.append(grouped.mean(dim=(1, 2)).cpu())

        return torch.stack(per_layer)

    finally:
        model.set_attn_implementation(old_backend)

a_raw = extract_raw_a()

assert torch.isfinite(a_raw).all(), "Scores contain NaN or infinity."
assert (a_raw >= 0).all(), "Attention scores contain negative values."
assert a_raw.shape == (
    model.config.num_hidden_layers,
    model.config.num_key_value_heads,
    len(doc_positions),
)

# 4. Save raw scores and token positions without modifying existing results.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
save_path = project / "results" / f"attention_a_probe_{stamp}.pt"

torch.save(
    {
        "case_id": case["case_id"],
        "document_id": case["document_id"],
        "first_question": question,
        "a_raw": a_raw,
        "document_positions": doc_positions,
        "question_positions": question_positions,
        "document_token_ids": document_token_ids,
        "source_environment": str(env_path),
        "model_id": model.config._name_or_path,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "transformers_version": transformers.__version__,
        "scoring_backend": "eager",
        "protected_positions_applied": False,
        "normalized": False,
    },
    save_path,
)

logger.debug('%s', "========== FIRST-QUESTION ATTENTION PROBE ==========")
logger.debug('%s %s', "Question:", question)
logger.debug('%s %s', "Total input tokens:", input_length)
logger.debug('%s %s', "Document tokens:", len(doc_positions))
logger.debug('%s %s', "First-question tokens:", len(question_positions))
logger.debug('%s %s', "a_raw shape:", tuple(a_raw.shape))
logger.debug('%s %s', "All values finite:", bool(torch.isfinite(a_raw).all()))
logger.debug('%s %s', "No negative values:", bool((a_raw >= 0).all()))
logger.debug('%s %s', "Restored attention backend:", model.config._attn_implementation)
logger.info('%s %s', "Saved to:", save_path)

## Original project cell 11


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import torch

# Reuse variables created by the attention-a cell.
required = [
    "model", "tokenizer", "render", "document", "prefix_text",
    "doc_positions", "document_token_ids", "a_raw", "case", "save_path",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Required variables from the attention-a cell are missing: {missing}"
    )

source_a_path = Path(save_path)
if not source_a_path.name.startswith("attention_a_probe_"):
    raise RuntimeError("save_path must refer to the preceding a-probe file.")

# 1. Replace the first question with a fixed context-repetition instruction.
# Do not pass reference answers, target questions, or condition labels to the model.
reconstruction_instruction = "Repeat the previous context exactly."
request_text = render(tokenizer, document, reconstruction_instruction)

if not request_text.startswith(prefix_text):
    raise ValueError("The shared document prefix has changed.")

prefix_ids = tokenizer(
    prefix_text,
    add_special_tokens=False,
    return_tensors="pt",
)["input_ids"]

request_ids = tokenizer(
    request_text,
    add_special_tokens=False,
    return_tensors="pt",
)["input_ids"]

prefix_len = prefix_ids.shape[1]

# Verify the document token positions and contents against the a-score input.
assert torch.equal(request_ids[:, :prefix_len], prefix_ids)
assert max(doc_positions) < prefix_len
assert request_ids[0, doc_positions].tolist() == document_token_ids

# 2. Replay the known document tokens at the start of the assistant response.
# Use a teacher-forced forward pass, not free document generation.
replay_ids = torch.tensor([document_token_ids], dtype=torch.long)
replay_start = request_ids.shape[1]
all_ids = torch.cat([request_ids, replay_ids], dim=1)

total_tokens = all_ids.shape[1]
if total_tokens > 1024:
    raise ValueError(
        "This diagnostic supports at most 1024 input tokens. "
        "Long documents are not silently truncated."
    )

# Reconstruction query range:
# Repetition instruction + trailing chat template + replayed document
query_start = prefix_len
n_queries = total_tokens - query_start

def extract_reconstruction_probe():
    old_backend = model.config._attn_implementation
    hq = model.config.num_attention_heads
    hkv = model.config.num_key_value_heads
    assert hq % hkv == 0

    inputs = all_ids.to(model.device)
    d_idx = torch.tensor(
        doc_positions, dtype=torch.long, device=model.device
    )

    model.eval()
    try:
        model.set_attn_implementation("eager")

        with torch.no_grad():
            output = model(
                input_ids=inputs,
                attention_mask=torch.ones_like(inputs),
                use_cache=False,
                output_attentions=True,
                return_dict=True,
                logits_to_keep=1,
            )

        if not output.attentions or any(
            weights is None for weights in output.attentions
        ):
            raise RuntimeError("Attention weights were not returned.")

        scores = []
        for weights in output.attentions:
            # Score only the keys of the original document.
            # Do not confuse replayed-document keys with original-document keys.
            block = (
                weights[0, :, query_start:, :]
                .index_select(2, d_idx)
                .float()
            )

            # [KV heads, grouped query heads, reconstruction queries, original document tokens]
            grouped = block.reshape(
                hkv, hq // hkv, n_queries, len(doc_positions)
            )

            # Unlike a, take the maximum across grouped query heads and query positions.
            scores.append(grouped.amax(dim=(1, 2)).cpu())

        return torch.stack(scores)

    finally:
        model.set_attn_implementation(old_backend)

g_raw_probe = extract_reconstruction_probe()

# 3. Validate numerical values and token alignment.
assert g_raw_probe.shape == a_raw.shape
assert torch.isfinite(g_raw_probe).all()
assert (g_raw_probe >= 0).all()
assert (g_raw_probe.sum(dim=-1) > 0).all()

# 4. Save separately without overwriting the a file or Full-KV results.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
g_save_path = (
    Path("/mnt/gstar-project/results")
    / f"attention_g_probe_{stamp}.pt"
)

torch.save(
    {
        "document_id": case["document_id"],
        "g_raw_probe": g_raw_probe,
        "document_positions": list(doc_positions),
        "document_token_ids": list(document_token_ids),
        "input_ids": all_ids.cpu(),
        "query_start": query_start,
        "replay_start": replay_start,
        "reconstruction_instruction": reconstruction_instruction,
        "source_a_probe": str(source_a_path),
        "model_id": model.config._name_or_path,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "scoring_backend": "eager",
        "score_definition": "single_chunk_full_causal_reconstruction_max",
        "native_kvzip_reproduction": False,
        "protected_positions_applied": False,
        "normalized": False,
    },
    g_save_path,
)

logger.debug('%s', "========== RECONSTRUCTION ATTENTION PROBE ==========")
logger.debug('%s %s', "Original document tokens:", len(doc_positions))
logger.debug('%s %s', "Total input tokens:", total_tokens)
logger.debug('%s %s', "Reconstruction query tokens:", n_queries)
logger.debug('%s %s', "g_raw_probe shape:", tuple(g_raw_probe.shape))
logger.debug('%s %s', "Shape matches a:", g_raw_probe.shape == a_raw.shape)
logger.debug('%s %s', "All values finite:", bool(torch.isfinite(g_raw_probe).all()))
logger.debug('%s %s', "No negative values:", bool((g_raw_probe >= 0).all()))
logger.debug('%s %s', "Every layer/head has positive score mass:", bool((g_raw_probe.sum(dim=-1) > 0).all()))
logger.debug('%s %s', "Restored attention backend:", model.config._attn_implementation)
logger.info('%s %s', "Saved to:", g_save_path)

## Original project cell 12


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")
SCRIPT = PROJECT / "gstar_mask_smoke.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the script to this path first:\n{SCRIPT}"
    )

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the Qwen model-loading cell first."
    )

smoke = runpy.run_path(str(SCRIPT))

mask_summary = smoke["run_mask_smoke"](
    model=model,
    tokenizer=tokenizer,
    project_dir=PROJECT,
    keep_fraction=0.5,
    max_new_tokens=64,
)

## Original project cell 13


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")
SCRIPT = PROJECT / "gstar_paired_pilot.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the script first:\n{SCRIPT}"
    )

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "The current kernel must contain model/tokenizer. "
        "If the kernel was restarted, run the model-loading cell first."
    )

paired = runpy.run_path(str(SCRIPT))

paired_summary = paired["run_paired_pilot"](
    model=model,
    tokenizer=tokenizer,
    project_dir=PROJECT,
    max_documents=3,
    keep_fraction=0.5,
    max_new_tokens=64,
)

## Original project cell 14


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

project = Path("/mnt/gstar-project")
run_dir = (
    project / "results"
    / "paired_pilot_20261022T134350_570386Z"
)

filenames = [
    "predictions.jsonl",
    "summary.json",
    "diagnostics.jsonl",
    "paired_targets.json",
    "environment.json",
]

missing = [name for name in filenames if not (run_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Result files not found: {missing}\nChecked directory: {run_dir}"
    )

archive = project / f"{run_dir.name}_review.zip"

with ZipFile(archive, "w", compression=ZIP_DEFLATED) as z:
    for name in filenames:
        z.write(run_dir / name, arcname=name)

    # Include data for reviewing questions alongside source documents.
    for name in ["selection_inputs.jsonl", "evaluation_only.jsonl"]:
        path = run_dir / "data_snapshot" / name
        if path.is_file():
            z.write(path, arcname=f"data_snapshot/{name}")

logger.info('%s %s', "Review ZIP:", archive)

## Original project cell 15


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")
SCRIPT = PROJECT / "gstar_expanded_eval.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the new script first:\n{SCRIPT}"
    )

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the model-loading cell first."
    )

expanded = runpy.run_path(str(SCRIPT))

EXP_DIR = expanded["prepare_experiment"](
    model=model,
    tokenizer=tokenizer,
    project_dir=PROJECT,
    n_dev=20,
    n_heldout=20,
)

logger.debug('%s %s', "\nExperiment directory:", EXP_DIR)
logger.debug('%s %s', "Development input preview:", Path(EXP_DIR) / "data" / "development" / "preview.txt")

## Original project cell 16


In [ ]:
dev_summary = expanded["run_development"](
    model=model,
    tokenizer=tokenizer,
    experiment_dir=EXP_DIR,
    project_dir=PROJECT,
    max_new_documents=None,  # Process all unfinished documents.
)

## Original project cell 17


In [ ]:
from pathlib import Path
import runpy
import shutil

PROJECT = Path("/mnt/gstar-project")
EXP_DIR = (
    PROJECT / "experiments"
    / "paired_span_v2_d20_h20_49366cdc1823"
)
SCRIPT = PROJECT / "gstar_expanded_eval.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(f"Script not found: {SCRIPT}")

if not (EXP_DIR / "runs" / "development" / "summary.json").is_file():
    raise FileNotFoundError(
        f"Development results not found:\n{EXP_DIR}"
    )

# Archive saved results only; no model inference.
expanded = runpy.run_path(str(SCRIPT))
review_zip = Path(
    expanded["export_review"](
        experiment_dir=EXP_DIR,
        split="development",
    )
)

# Also copy to the project root for convenient access.
download_path = PROJECT / review_zip.name
shutil.copy2(review_zip, download_path)

logger.debug('%s', "\nDownload file:")
logger.debug('%s', download_path)
logger.debug('%s', f"File size: {download_path.stat().st_size / 1024**2:.2f} MiB")

## Original project cell 18


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")
EXP_DIR = (
    PROJECT / "experiments"
    / "paired_span_v2_d20_h20_49366cdc1823"
)
SCRIPT = PROJECT / "gstar_expanded_eval.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(f"Script is missing: {SCRIPT}")

if not (EXP_DIR / "runs" / "development" / "summary.json").is_file():
    raise FileNotFoundError(f"Development results are missing: {EXP_DIR}")

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the model-loading cell first."
    )

expanded = runpy.run_path(str(SCRIPT))

# Freeze the settings selected on development data.
expanded["freeze_protocol"](experiment_dir=EXP_DIR)

# Use frozen settings only; do not retune on held-out data.
heldout_summary = expanded["run_heldout"](
    model=model,
    tokenizer=tokenizer,
    experiment_dir=EXP_DIR,
    project_dir=PROJECT,
    max_new_documents=None,
)

## Original project cell 19


In [ ]:
import shutil

review_zip = Path(
    expanded["export_review"](
        experiment_dir=EXP_DIR,
        split="heldout",
    )
)

download_path = PROJECT / review_zip.name
shutil.copy2(review_zip, download_path)

logger.debug('%s %s', "Download file:", download_path)

## Original project cell 20


In [ ]:
loaded_model = globals().get("model")
loaded_tokenizer = globals().get("tokenizer")

if loaded_model is None or loaded_tokenizer is None:
    logger.debug('%s', "The current kernel has no model/tokenizer.")
    logger.debug('%s', "Run the model-loading cell first.")
else:
    logger.debug('%s', "The model and tokenizer are still loaded.")
    logger.debug('%s %s', "Model:", loaded_model.config._name_or_path)
    logger.debug('%s %s', "Device:", loaded_model.device)
    logger.debug('%s %s', "dtype:", loaded_model.dtype)
    logger.debug('%s', "Continue without reloading the model.")

## Original project cell 21


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")

# Source experiment containing completed development and held-out runs
EXP_DIR = (
    PROJECT / "experiments"
    / "paired_span_v2_d20_h20_49366cdc1823"
)

SCRIPT = PROJECT / "gstar_extract_coverage.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the new script first:\n{SCRIPT}"
    )

if not (EXP_DIR / "protocol.json").is_file():
    raise FileNotFoundError(
        f"Source experiment directory not found:\n{EXP_DIR}"
    )

if globals().get("model") is None or globals().get("tokenizer") is None:
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the model-loading cell first."
    )

coverage_runner = runpy.run_path(str(SCRIPT))

coverage_summary = coverage_runner["extract_coverage"](
    model=model,
    tokenizer=tokenizer,
    experiment_dir=EXP_DIR,
    document_ids=["development_001"],  # Start with one document.
    n_blocks=4,
)

## Original project cell 22


In [ ]:
coverage_summary = coverage_runner["extract_coverage"](
    model=model,
    tokenizer=tokenizer,
    experiment_dir=EXP_DIR,
    document_ids=None,  # All original development documents.
    n_blocks=4,        # Use the same four blocks as in the first document.
)

## Original project cell 23


In [ ]:
EXP_DIR = (
    PROJECT / "experiments"
    / "paired_span_v2_d20_h20_49366cdc1823"
)

COV_DIR = (
    PROJECT / "experiments"
    / "coverage_blocks_v1_b4_53450bb3116a"
)

SCRIPT = PROJECT / "gstar_coverage_eval.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the new file first:\n{SCRIPT}"
    )

coverage_eval = runpy.run_path(str(SCRIPT))

SELECTOR_DIR = coverage_eval["prepare_coverage_selectors"](
    source_experiment_dir=EXP_DIR,
    coverage_dir=COV_DIR,
)

## Original project cell 24


In [ ]:
if globals().get("model") is None or globals().get("tokenizer") is None:
    raise RuntimeError(
        "The current kernel must contain model/tokenizer. "
        "No reload is needed if the model is still loaded."
    )

coverage_dev = coverage_eval["run_coverage_development"](
    model=model,
    tokenizer=tokenizer,
    selector_dir=SELECTOR_DIR,
    max_new_documents=1,
)

## Original project cell 25


In [ ]:
coverage_dev = coverage_eval["run_coverage_development"](
    model=model,
    tokenizer=tokenizer,
    selector_dir=SELECTOR_DIR,
    max_new_documents=None,
)

## Original project cell 27


In [ ]:
from pathlib import Path
import runpy

PROJECT = Path("/mnt/gstar-project")

SOURCE_EXP = (
    PROJECT / "experiments"
    / "paired_span_v2_d20_h20_49366cdc1823"
)

SCRIPT = PROJECT / "gstar_fresh_coverage_test.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Upload the new script first:\n{SCRIPT}"
    )

if globals().get("model") is None or globals().get("tokenizer") is None:
    raise RuntimeError(
        "The current kernel has no model/tokenizer. "
        "Run the existing model-loading cell or the optional loading cell in the README."
    )

fresh = runpy.run_path(str(SCRIPT))

TEST_DIR = fresh["prepare_fresh_test"](
    model=model,
    tokenizer=tokenizer,
    source_experiment_dir=SOURCE_EXP,
    n_documents=20,
)

## Original project cell 28


In [ ]:
fresh_summary = fresh["run_fresh_test"](
    model=model,
    tokenizer=tokenizer,
    test_dir=TEST_DIR,
    max_new_documents=None,
)

## Original project cell 29


In [ ]:
import shutil

zip_path = Path(
    fresh["export_review"](
        test_dir=TEST_DIR,
    )
)

download_path = PROJECT / zip_path.name
shutil.copy2(zip_path, download_path)

logger.debug('%s', "Download ZIP:")
logger.debug('%s', download_path)